# 금천구 무더위쉼터 추출

`서울시_무더위쉼터.csv`에서 금천구에 해당하는 쉼터만 추려내고,
좌표를 정리해 이후 접근성 분석에 바로 쓸 수 있는 형태로 저장한다.

원본은 서울 열린데이터광장 서울시 무더위쉼터(OA-21065) 자료이며 인코딩은 cp949다.
좌표는 경도/위도(WGS84)와 EPSG:5186 두 가지가 들어 있다.

## 1. 자료 불러오기

In [ ]:
import pandas as pd
from pathlib import Path

# 원본 CSV는 2_데이터/원본 폴더에 둔다.
# 다른 곳에 있으면 아래 경로만 바꾼다.
CSV_PATH = Path("../2_데이터/원본/서울시_무더위쉼터.csv")

# 원본 인코딩은 cp949다. utf-8로 읽으면 실패한다.
df = pd.read_csv(CSV_PATH, encoding="cp949", dtype=str)

print("전체 행 수:", len(df))
print("컬럼 수:", df.shape[1])
print()
print(df.columns.tolist())

In [ ]:
# 시설년도 확인. 이 자료가 어느 해 기준인지 먼저 본다.
print(df["시설년도"].value_counts())

df.head(3)

## 2. 금천구 필터링

금천구를 골라내는 방법이 세 가지 있고, 결과가 서로 다르다.

- 위치코드 앞 5자리가 `11545` (금천구 시군구 코드)
- 지번주소에 "금천구" 포함
- 도로명주소에 "금천구" 포함

세 방법을 각각 세어보고 차이를 확인한 뒤 합집합을 쓴다.
한 가지 방법만 쓰면 누락이 생긴다.

In [ ]:
GEUMCHEON_CODE = "11545"   # 서울특별시 금천구 시군구 코드

by_code = df["위치코드"].fillna("").str[:5] == GEUMCHEON_CODE
by_jibun = df["지번주소"].fillna("").str.contains("금천구")
by_road = df["도로명주소"].fillna("").str.contains("금천구")

print("위치코드 기준 :", by_code.sum(), "건")
print("지번주소 기준 :", by_jibun.sum(), "건")
print("도로명주소 기준:", by_road.sum(), "건")

is_geumcheon = by_code | by_jibun | by_road
print()
print("합집합        :", is_geumcheon.sum(), "건")

In [ ]:
# 방법에 따라 결과가 갈리는 행을 직접 확인한다.
cols = ["위치코드", "쉼터명칭", "도로명주소", "지번주소"]

print("[코드는 금천구가 아닌데 주소는 금천구]")
print(df.loc[is_geumcheon & ~by_code, cols].to_string(index=False))

print()
print("[코드는 금천구인데 도로명주소가 비어 있음]")
print(df.loc[by_code & df["도로명주소"].isna(), cols].to_string(index=False))

확인해 보면 두 가지 경우가 나온다.

하나는 위치코드가 `1100000000`(서울시 본청)으로 등록되어 있지만 주소는 금천구인 시설이다.
자치구가 아니라 시에서 직접 운영하는 시설이라 코드가 다르게 붙은 것으로 보인다.
코드만으로 거르면 이 시설이 빠진다.

다른 하나는 도로명주소가 비어 있고 지번주소만 있는 시설이다.
도로명주소만으로 거르면 이 시설들이 빠진다.

그래서 합집합을 쓴다.

In [ ]:
geumcheon = df.loc[is_geumcheon].copy().reset_index(drop=True)

print("금천구 무더위쉼터:", len(geumcheon), "개소")
geumcheon.head()

## 3. 값 정리

원본에 두 가지 문제가 있다.

- `시설구분2`의 가운뎃점이 원본 단계에서 이미 `?`로 깨져 있다. 예: `복지?문화?체육시설`
- 좌표가 문자열이라 숫자로 바꿔야 한다

In [ ]:
# 원본에 이미 깨져 들어온 물음표를 가운뎃점으로 되돌린다.
geumcheon["시설구분2"] = geumcheon["시설구분2"].str.replace("?", "·", regex=False)

# 좌표를 숫자로 변환한다.
num_cols = ["경도", "위도", "X좌표(EPSG:5186)", "Y좌표(EPSG:5186)"]
for c in num_cols:
    geumcheon[c] = pd.to_numeric(geumcheon[c], errors="coerce")

print("좌표 결측 건수")
print(geumcheon[num_cols].isna().sum())

In [ ]:
# 좌표가 서울 범위 안에 있는지 확인한다. 범위를 벗어나면 지오코딩 오류다.
lon_ok = geumcheon["경도"].between(126.7, 127.3)
lat_ok = geumcheon["위도"].between(37.4, 37.7)

bad = geumcheon.loc[~(lon_ok & lat_ok)]
if len(bad) == 0:
    print("좌표 이상치 없음")
else:
    print("좌표 이상치", len(bad), "건")
    print(bad[["쉼터명칭", "지번주소", "경도", "위도"]].to_string(index=False))

## 4. 구성 살펴보기

In [ ]:
print("[시설구분1]")
print(geumcheon["시설구분1"].value_counts())
print()
print("[시설구분2]")
print(geumcheon["시설구분2"].value_counts())

## 5. 폭염 피크 시간대에 실제로 여는 곳

지정 개소 수와 실제로 갈 수 있는 개소 수는 다르다.
평일 오후 1시부터 5시까지를 폭염 피크 시간대로 보고,
그 시간에 문을 여는 쉼터가 몇 개인지 센다.

이 숫자가 뒤에서 커버율을 계산할 때 기준이 된다.

In [ ]:
def to_minutes(s):
    """'09:00' 형태를 분 단위 정수로. 결측이면 None."""
    if pd.isna(s) or str(s).strip() == "":
        return None
    try:
        h, m = str(s).split(":")[:2]
        return int(h) * 60 + int(m)
    except ValueError:
        return None


PEAK_START = 13 * 60   # 13:00
PEAK_END = 17 * 60     # 17:00
WEEKDAYS = {"월", "화", "수", "목", "금"}

start = geumcheon["기본운영_시작시간"].map(to_minutes)
end = geumcheon["기본운영_종료시간"].map(to_minutes)

# 08:00~02:00 처럼 자정을 넘겨 운영하는 곳이 있다.
# 그대로 두면 종료시간(120분)이 시작시간(480분)보다 작아 미개방으로 잘못 분류된다.
overnight = end.notna() & start.notna() & (end <= start)
end = end.where(~overnight, end + 24 * 60)
print("자정을 넘겨 운영하는 시설:", overnight.sum(), "개소")

# 기본운영 요일에 평일이 하나라도 들어있는지
has_weekday = geumcheon["기본운영_요일"].fillna("").apply(
    lambda s: bool(WEEKDAYS & set(str(s).split(",")))
)

geumcheon["피크시간_개방"] = (
    has_weekday
    & start.notna() & end.notna()
    & (start <= PEAK_START)
    & (end >= PEAK_END)
)

print("전체 금천구 쉼터      :", len(geumcheon), "개소")
print("평일 13~17시 개방     :", geumcheon["피크시간_개방"].sum(), "개소")
print("피크시간 미개방       :", (~geumcheon["피크시간_개방"]).sum(), "개소")

In [ ]:
# 주말 개방 여부도 같이 본다.
WEEKEND = {"토", "일"}

geumcheon["주말_개방"] = geumcheon["기본운영_요일"].fillna("").apply(
    lambda s: bool(WEEKEND & set(str(s).split(",")))
)

# 연장운영이나 추가운영이 있는 곳
geumcheon["연장운영"] = geumcheon["연장운영_운영여부"].fillna("N").str.upper().eq("Y")
geumcheon["추가운영"] = geumcheon["추가운영_운영여부"].fillna("N").str.upper().eq("Y")

summary = pd.DataFrame({
    "개소수": [
        len(geumcheon),
        geumcheon["피크시간_개방"].sum(),
        geumcheon["주말_개방"].sum(),
        geumcheon["연장운영"].sum(),
        geumcheon["추가운영"].sum(),
    ]
}, index=["전체 지정", "평일 13~17시 개방", "주말 개방", "연장운영 있음", "추가운영 있음"])

summary["비율(%)"] = (summary["개소수"] / len(geumcheon) * 100).round(1)
summary

In [ ]:
# 시설구분별로 피크시간 개방률을 본다.
pivot = pd.crosstab(
    geumcheon["시설구분1"],
    geumcheon["피크시간_개방"],
)
pivot.columns = ["미개방", "개방"]
pivot["개방률(%)"] = (pivot["개방"] / pivot.sum(axis=1) * 100).round(1)
pivot

## 6. 저장

In [ ]:
OUT_CSV = Path("../2_데이터/가공/금천구_무더위쉼터.csv")

keep = [
    "시설년도", "위치코드", "시설구분1", "시설구분2", "쉼터명칭",
    "도로명주소", "지번주소",
    "기본운영_요일", "기본운영_시작시간", "기본운영_종료시간",
    "피크시간_개방", "주말_개방", "연장운영", "추가운영",
    "경도", "위도", "X좌표(EPSG:5186)", "Y좌표(EPSG:5186)",
]

geumcheon[keep].to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("저장 완료:", OUT_CSV.resolve())
print("행 수:", len(geumcheon))

## 7. 공간 자료로 변환

여기부터는 geopandas가 필요하다. 설치되어 있지 않으면 이 아래는 건너뛰어도
6번까지의 CSV는 그대로 쓸 수 있다.

격자 인구 자료가 EPSG:5179이므로 쉼터도 같은 좌표계로 맞춘다.
원본의 EPSG:5186 좌표 대신 경도/위도에서 바로 변환한다.

In [ ]:
try:
    import geopandas as gpd
    HAS_GPD = True
except ImportError:
    HAS_GPD = False
    print("geopandas가 없다. 아래 셀은 건너뛴다.")
    print("설치: conda install -c conda-forge geopandas")

In [ ]:
if HAS_GPD:
    shelters = gpd.GeoDataFrame(
        geumcheon[keep],
        geometry=gpd.points_from_xy(geumcheon["경도"], geumcheon["위도"]),
        crs="EPSG:4326",
    )

    # 격자 인구 자료와 같은 좌표계로 변환
    shelters = shelters.to_crs("EPSG:5179")

    print("좌표계:", shelters.crs)
    print("개소:", len(shelters))
    print()
    print(shelters.total_bounds)   # minx, miny, maxx, maxy

In [ ]:
if HAS_GPD:
    OUT_GPKG = Path("../2_데이터/가공/금천구_무더위쉼터.gpkg")
    shelters.to_file(OUT_GPKG, layer="shelters", driver="GPKG")
    print("저장 완료:", OUT_GPKG.resolve())

## 8. 확인용 지도

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# 한글이 깨지지 않도록 폰트를 지정한다.
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(9, 9))

for opened, group in geumcheon.groupby("피크시간_개방"):
    ax.scatter(
        group["경도"], group["위도"],
        s=45,
        alpha=0.8,
        label="평일 13~17시 개방" if opened else "피크시간 미개방",
    )

ax.set_title(f"금천구 무더위쉼터 {len(geumcheon)}개소")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
ax.legend()
ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout()
plt.show()

## 다음 단계

여기까지가 시설 쪽 준비다. 이어서 할 일은 다음과 같다.

1. 서울시 자치구별 도보 네트워크(OA-21208)에서 금천구 구간을 잘라 그래프로 만든다
2. 쉼터 좌표를 네트워크 노드에 스냅한다
3. 격자 수요지에서 각 쉼터까지 최단 보행거리를 구한다
4. 기준거리 안에 들어오는 인구 비율, 즉 기존 배치의 커버율을 계산한다

4번을 두 번 계산해 두면 좋다.
전체 지정 쉼터로 한 번, 피크시간에 실제로 여는 쉼터로 한 번이다.
두 숫자의 차이가 이 분석의 출발점이 된다.